In [ ]:
%pip install scikit-learn

In [ ]:
import pandas as pd
import numpy as np


importing tools to do the train test split, make the pipeline, column transformer to easily convert the categorical and numerical features into onehot encoding and scaling repectively

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

lets load the csv file now

In [ ]:
df=pd.read_csv("bank-additional-full.csv", sep=';')

In [ ]:
df.shape

In [ ]:
df.info()

so there are no missing values at all

In [ ]:
df.duplicated().sum()

there are 12 duplicated rows in here. lets see them

In [ ]:
duplicates = df[df.duplicated]

In [ ]:
duplicates

nothing wrong with the duplicated rows so ill drop them. 

In [ ]:
df=df.drop_duplicates()

In [ ]:
df.shape

now lets see if our data is imbalanced or not 

In [ ]:
df["y"].value_counts()

There are 36537 nos and only 4639 yes'. data is imbalanced

i will handle the imbalanced dataset after splitting; not now. 

In [ ]:
df.describe()

now i will look at the features that we have : 
"age" "job" "marital" "education" "default" "housing" "loan" "contact" "month" "day_of_week" "duration" "campaign" "pdays" "previous" "poutcome" "emp.var.rate" "cons.price.idx" "cons.conf.idx" "euribor3m" "nr.employed" "y"

here duration represents the length of call that happened when the customer was called last time. here we are deciding  which customers to call before making the call. we cannot know the duration of the call before actually making it. 

also it is highly misleading, because the duration with 0 seconds willl always have answer 'no'

therefore we drop duplicates

In [ ]:
df = df.drop(columns=['duration'])

In [ ]:
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

In [ ]:
numeric_cols

In [ ]:
categorical_cols

In [ ]:
for col in categorical_cols:
    print(df[col].value_counts())
    print('\n')

there are unknown values here. but i will consider them as a separate cztegory rather than removing them or imputing anything else. 

pdays:

In [ ]:
print(df['pdays'].value_counts())

999 pdays means customers were not contacted before. and there are ALOT of them

so what i will do is i will add onemore column named contactedaBefore which will be a binary column. it will have 1 if contacted before else 0 (pdays=999)

In [ ]:
df['contacted_before'] = np.where(df['pdays'] == 999, 0, 1)


after doing this , we have separated the cusstomers into contactedbefore or not. but we need to chnage our og pdays column as well. we will change all the 999 values to 0. and keeping rest of it as it is. so the workflow will look something like: model will look if it was contacted before, from the new added column. if yes, thenn it will look howmany days ago from pdays column

In [ ]:

df['pdays'] = np.where(df['pdays'] == 999, 0, df['pdays'])


In [ ]:
#updating our numeric cols and categorical cols 
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()